# FBSA — Evaluation & Visualization

Pick an arch via `ARCH` (top of cell-1):
- `fbsa`        — v1 (slot only)
- `fbsa_fused`  — v2 (option 1: slot + encoder fusion)
- `fbsa_skip`   — v3 (option 2: + image-stem multi-scale skips)

Loads `best_model.pt` from that arch's run dir, computes test metrics, and shows
`image | gt | pred | TP/FP/FN overlay` per sample. The overlay is what reveals
boundary quality — Dice can be the same while one model has clean edges and the
other has ragged ones. Optional final cell does a side-by-side comparison across
every arch whose checkpoint is on disk.

In [ ]:
import sys, random
from pathlib import Path
PROJECT = Path('..').resolve()
sys.path.insert(0, str(PROJECT))
import numpy as np
import torch, kagglehub, matplotlib.pyplot as plt
from tumor_seg.config import TrainConfig
from tumor_seg.data import create_brisc_dataloaders
from tumor_seg.metrics import dice_coefficient, iou_coefficient, hausdorff_distance_metric
from tumor_seg.models import build_model

ARCH = 'fbsa_skip'                                            # ← v3
RUN_DIR = f'/content/drive/MyDrive/fbsa_runs/{ARCH}'
ckpt_path = f'{RUN_DIR}/best_model.pt'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
cfg = TrainConfig(**ckpt['cfg'])
model = build_model(cfg).to(device)
model.load_state_dict(ckpt['model'])
model.eval()
print(f'arch = {cfg.arch}  best epoch = {ckpt["epoch"]}')

In [ ]:
data_root = str(Path(kagglehub.dataset_download('briscdataset/brisc2025')) / 'brisc2025' / 'segmentation_task')
_, val_loader = create_brisc_dataloaders(data_root, batch_size=cfg.batch_size, image_size=cfg.image_size)

tot_d = tot_i = tot_h = 0.0; n = 0
all_imgs, all_gts, all_preds = [], [], []
with torch.no_grad():
    for images, masks in val_loader:
        images, masks = images.to(device), masks.to(device)
        logits = model(images)
        bs = images.size(0); n += bs
        tot_d += dice_coefficient(logits, masks).item() * bs
        tot_i += iou_coefficient(logits, masks).item() * bs
        tot_h += hausdorff_distance_metric(logits, masks, image_size=cfg.image_size) * bs
        all_imgs.append(images.cpu()); all_gts.append(masks.cpu())
        all_preds.append((torch.sigmoid(logits) > 0.5).float().cpu())
print(f'[{cfg.arch}]  Dice = {tot_d/n:.4f}  IoU = {tot_i/n:.4f}  HD = {tot_h/n:.2f} px')

In [ ]:
# Per-image visualization: image | gt | pred | TP/FP/FN overlay
# Overlay legend: green=TP (correct tumor), red=FP (false alarm), blue=FN (missed tumor)
imgs = torch.cat(all_imgs); gts = torch.cat(all_gts); preds = torch.cat(all_preds)

def make_error_overlay(image_chw, gt_hw, pred_hw):
    img = image_chw.permute(1, 2, 0).numpy()
    img = (img - img.min()) / (img.max() - img.min() + 1e-8)
    base = (img.mean(axis=2, keepdims=True).repeat(3, axis=2)) * 0.6
    gt = gt_hw.numpy() > 0.5
    pr = pred_hw.numpy() > 0.5
    overlay = base.copy()
    overlay[gt &  pr] = [0.1, 0.9, 0.1]    # TP green
    overlay[~gt &  pr] = [0.95, 0.2, 0.2]  # FP red
    overlay[gt & ~pr] = [0.2, 0.4, 0.95]   # FN blue
    return img, overlay

k = min(10, imgs.shape[0])
random.seed(0)
idxs = random.sample(range(imgs.shape[0]), k)
fig, axes = plt.subplots(k, 4, figsize=(12, 3 * k))
if k == 1:
    axes = np.expand_dims(axes, 0)
for r, i in enumerate(idxs):
    img, overlay = make_error_overlay(imgs[i], gts[i, 0], preds[i, 0])
    axes[r, 0].imshow(img);                         axes[r, 0].set_title('image')
    axes[r, 1].imshow(gts[i, 0], cmap='gray');      axes[r, 1].set_title('gt')
    axes[r, 2].imshow(preds[i, 0], cmap='gray');    axes[r, 2].set_title('pred')
    axes[r, 3].imshow(overlay);                     axes[r, 3].set_title('TP / FP / FN')
    for j in range(4):
        axes[r, j].axis('off')
plt.suptitle(f'{cfg.arch}  (epoch {ckpt["epoch"]})', y=1.001)
plt.tight_layout(); plt.show()

## Optional — side-by-side comparison across arches

Loads every checkpoint that exists under `fbsa_runs/{arch}/best_model.pt` and
renders one row per sample with `image | gt | pred_v1 | pred_v2 | pred_v3`.
Skip this cell if you only have one checkpoint trained.

In [ ]:
from tumor_seg.models import ARCH_REGISTRY

available = []
for arch_key in ARCH_REGISTRY:
    p = Path(f'/content/drive/MyDrive/fbsa_runs/{arch_key}/best_model.pt')
    if p.exists():
        available.append((arch_key, p))
print('checkpoints found:', [a for a, _ in available])

loaded = {}  # arch_key -> (model, cfg)
for arch_key, p in available:
    c = torch.load(p, map_location=device, weights_only=False)
    cfg_a = TrainConfig(**c['cfg'])
    m = build_model(cfg_a).to(device); m.load_state_dict(c['model']); m.eval()
    loaded[arch_key] = (m, cfg_a)

# Run each model on the same val batch (use first val batch we cached above)
comp_imgs = imgs[:k]; comp_gts = gts[:k]
preds_per_arch = {}
with torch.no_grad():
    for arch_key, (m, _) in loaded.items():
        logits = m(comp_imgs.to(device))
        preds_per_arch[arch_key] = (torch.sigmoid(logits) > 0.5).float().cpu()

ncols = 2 + len(preds_per_arch)
fig, axes = plt.subplots(k, ncols, figsize=(3 * ncols, 3 * k))
if k == 1:
    axes = np.expand_dims(axes, 0)
for r in range(k):
    img = comp_imgs[r].permute(1, 2, 0).numpy()
    img = (img - img.min()) / (img.max() - img.min() + 1e-8)
    axes[r, 0].imshow(img);                         axes[r, 0].set_title('image')
    axes[r, 1].imshow(comp_gts[r, 0], cmap='gray'); axes[r, 1].set_title('gt')
    for c, arch_key in enumerate(preds_per_arch):
        axes[r, 2 + c].imshow(preds_per_arch[arch_key][r, 0], cmap='gray')
        axes[r, 2 + c].set_title(arch_key)
    for j in range(ncols):
        axes[r, j].axis('off')
plt.tight_layout(); plt.show()